#### Suppose we have a list of 50 candidate items with predicted relevance scores from a model. Write pseudocode to re-rank these items using MMR, ensuring at least 5 distinct genres are represented in the top-10.

In [42]:
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional

In [9]:
def _cosine_similarity_matrix(embs: np.ndarray) -> np.ndarray:
    # embs: (n, d)
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    norms[norms == 0] = 1e-8
    embs_norm = embs / norms
    return embs_norm @ embs_norm.T

def rerank_with_genre_mmr(candidates: pd.DataFrame, top_k: int = 10, desired_genres: int = 5,
                         lambda_param: float = 0.7, genre_boost: float = 0.15) -> pd.DataFrame:
    """Re-rank candidates using MMR while encouraging at least `desired_genres` in top_k.

    candidates: DataFrame with columns ['id', 'score', 'genre', 'embedding'] where 'embedding' is a 1-D
                numpy array for each row.
    Returns selected rows in ranking order (DataFrame of length <= top_k).
    """
    if candidates.empty:
        return candidates.copy()

    # Ensure embeddings are stacked into a numpy array (n, d)
    embs_list = list(candidates['embedding'].values)
    embs = np.vstack([np.asarray(e, dtype=np.float32).reshape(1, -1) for e in embs_list])
    n = embs.shape[0]

    # Normalize model scores to [0,1] (min-max). If all equal, set to 1.
    scores = np.asarray(candidates['score'].values, dtype=np.float32)
    smin, smax = scores.min(), scores.max()
    if smax > smin:
        norm_scores = (scores - smin) / (smax - smin)
    else:
        norm_scores = np.ones_like(scores)

    # Precompute similarity matrix (cosine)
    sim = _cosine_similarity_matrix(embs)  # shape (n,n), in [-1,1]

    # Setup selection state
    selected_idxs: List[int] = []
    remaining = set(range(n))
    selected_genres = set()

    all_genres = set(candidates['genre'].astype(str).tolist())
    if len(all_genres) < desired_genres:
        # fallback: reduce desired_genres to available distinct genres
        desired_genres = len(all_genres)

    # Iterative MMR selection with genre boost to encourage new genres early
    while len(selected_idxs) < top_k and remaining:
        best_idx = None
        best_score = -np.inf

        for idx in list(remaining):
            rel = float(norm_scores[idx])
            if selected_idxs:
                max_sim = float(np.max(sim[idx, selected_idxs]))
            else:
                max_sim = 0.0

            mmr_score = lambda_param * rel - (1.0 - lambda_param) * max_sim

            # If we still need to reach desired_genres, give a small boost to items that add a new genre
            genre = str(candidates.iloc[idx]['genre'])
            if len(selected_genres) < desired_genres and genre not in selected_genres:
                mmr_score += genre_boost

            # keep best
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx

        if best_idx is None:
            break

        # Select it
        selected_idxs.append(best_idx)
        remaining.remove(best_idx)
        selected_genres.add(str(candidates.iloc[best_idx]['genre']))

    # Return DataFrame of selected items in order
    selected_df = candidates.iloc[selected_idxs].copy().reset_index(drop=True)
    return selected_df

In [10]:
# Create 50 synthetic candidates with 10 possible genres
rng = np.random.RandomState(42)
n_candidates = 50
d = 32
genres = [f'genre_{i}' for i in range(10)]

rows = []
for i in range(n_candidates):
    emb = rng.randn(d).astype(np.float32)
    # give slightly higher scores to lower-index items to simulate model scores
    score = float(n_candidates - i) + rng.randn() * 0.1
    # assign genres in a biased way but ensure at least 6 genres appear
    genre = genres[i % len(genres)] if i % 7 != 0 else genres[(i*3) % len(genres)]
    rows.append({'id': i, 'score': score, 'genre': genre, 'embedding': emb})

df = pd.DataFrame(rows)

top = rerank_with_genre_mmr(df, top_k=10, desired_genres=5, lambda_param=0.7, genre_boost=0.2)
print('Top-10 IDs:', top['id'].tolist())
print('Top-10 genres:', top['genre'].tolist())
print('Distinct genres in top-10:', len(set(top['genre'].tolist())))

# Sanity check: ensure at least 5 distinct genres (or available-count fallback)
assert len(set(top['genre'].tolist())) >= 5, 'Did not reach desired genre diversity'
print('MMR re-rank demo passed (>=5 genres in top-10)')

Top-10 IDs: [0, 3, 6, 4, 1, 2, 8, 5, 7, 9]
Top-10 genres: ['genre_0', 'genre_3', 'genre_6', 'genre_4', 'genre_1', 'genre_2', 'genre_8', 'genre_5', 'genre_1', 'genre_9']
Distinct genres in top-10: 9
MMR re-rank demo passed (>=5 genres in top-10)


#### We want to incorporate a simple fairness re-ranking in a music recommendation list, so that at most 40% of the top-10 songs are by the same artist. Describe an algorithm or code approach to post-process an initial ranked list to enforce this constraint (e.g. by swapping or down-ranking items violating the rule).

In [38]:
import math
from collections import defaultdict, Counter

In [15]:
def enforce_artist_cap(ranked_items: List[Dict[str, Any]], top_k: int = 10, cap_fraction: float = 0.4, allow_fallback: bool = True) -> List[Dict[str, Any]]:
    """Post-process a ranked list so that at most `floor(cap_fraction*top_k)` items in the top_k share the same artist.

    ranked_items: list of dicts sorted by descending model score. Each dict must contain at least 'id' and 'artist'.
    Returns the selected top-K list (length <= top_k).
    """
    if top_k <= 0:
        return []

    max_per_artist = int(math.floor(cap_fraction * top_k))
    if max_per_artist == 0:
        # allow at least 1 per artist to be able to fill slots in most practical settings
        max_per_artist = 1

    remaining = list(ranked_items)  # preserve original order
    selected: List[Dict[str, Any]] = []
    artist_count = defaultdict(int)

    desired_len = min(top_k, len(ranked_items))

    for pos in range(desired_len):
        picked_idx = None
        for i, item in enumerate(remaining):
            artist = item.get('artist')
            if artist_count[artist] < max_per_artist:
                picked_idx = i
                break

        if picked_idx is None:
            if allow_fallback:
                picked_idx = 0
            else:
                break

        picked = remaining.pop(picked_idx)
        selected.append(picked)
        artist_count[picked.get('artist')] += 1

    return selected

In [12]:
musics = pd.read_csv('SpotifySongs.csv')

In [ ]:
ranked_items = []
# build a small ranked list from the CSV (take first 50 rows as candidates)
cand = musics.sample(50, random_state=42).sort_values(by='Popularity', ascending=False).reset_index(drop=True)
for i, row in cand.iterrows():
    ranked_items.append({'id': int(i), 'artist': str(row['ArtistName']), 'score': float(row['Popularity']), 'title': row['SongName']})

print('Initial top-10 artists (before enforcement):')
print([it['artist'] for it in ranked_items[:10]])

# Apply the cap (10% of 10 -> 1 max per artist)
selected = enforce_artist_cap(ranked_items, top_k=10, cap_fraction=0.1, allow_fallback=True)
artist_counts = Counter([it['artist'] for it in selected])
print('Artist counts in selected top-10:', dict(artist_counts))
print('Selected top-10 after artist-cap enforcement:')
for i, it in enumerate(selected, 1):
    print(f"{i}. id={it['id']}, artist={it['artist']}, score={it['score']:.3f}, title={it.get('title','')}")

Initial top-10 artists (before enforcement):
['Shontelle', 'Post Malone', 'Griff', 'Post Malone', 'Rihanna', 'Arijit Singh', 'Moore Kismet', 'Juice WRLD', 'The Chainsmokers', 'Johnny Orlando']
Artist counts in selected top-10: {'Shontelle': 1, 'Post Malone': 1, 'Griff': 1, 'Rihanna': 1, 'Arijit Singh': 1, 'Moore Kismet': 1, 'Juice WRLD': 1, 'The Chainsmokers': 1, 'Johnny Orlando': 1, 'Pyotr Ilyich Tchaikovsky': 1}
Selected top-10 after artist-cap enforcement:
1. id=0, artist=Shontelle, score=97.000, title=Impossible
2. id=1, artist=Post Malone, score=92.000, title=Congratulations
3. id=2, artist=Griff, score=90.000, title=Shade of Yellow
4. id=4, artist=Rihanna, score=89.000, title=Diamonds
5. id=5, artist=Arijit Singh, score=88.000, title=Ae Watan (Male)
6. id=6, artist=Moore Kismet, score=87.000, title=You Should Run
7. id=7, artist=Juice WRLD, score=87.000, title=734
8. id=8, artist=The Chainsmokers, score=87.000, title=Call You Mine
9. id=9, artist=Johnny Orlando, score=86.000, tit

#### Outline how we would implement Thompson Sampling for a recommendation scenario. For example, we have a Bayesian model that gives a distribution of predicted ratings for each item. How would Thompson Sampling select an item to recommend to a user in one round?

In [43]:
from dataclasses import dataclass

In [44]:
@dataclass
class BetaPosterior:
    """Posterior distribution for item ratings using Beta distribution."""
    alpha: float  # successes + 1 (like counts)
    beta: float   # failures + 1 (dislike counts)
    
    def sample(self) -> float:
        """Draw a sample from Beta(alpha, beta)."""
        return np.random.beta(self.alpha, self.beta)
    
    def mean(self) -> float:
        """Expected value of Beta(alpha, beta)."""
        return self.alpha / (self.alpha + self.beta)
    
    def update(self, success: bool) -> None:
        """Update posterior with a new observation."""
        if success:
            self.alpha += 1
        else:
            self.beta += 1

class ThompsonSamplingRecommender:
    """Thompson Sampling for recommendation using Beta posteriors per item."""
    
    def __init__(self, n_items: int, prior_alpha: float = 1.0, prior_beta: float = 1.0):
        """
        Args:
            n_items: Number of items in catalog
            prior_alpha: Prior alpha for all items (default: 1.0 for uniform prior)
            prior_beta: Prior beta for all items (default: 1.0 for uniform prior)
        """
        self.posteriors = [
            BetaPosterior(alpha=prior_alpha, beta=prior_beta)
            for _ in range(n_items)
        ]
    
    def recommend(self, k: int = 10, exploit: bool = False) -> List[Tuple[int, float]]:
        """Get top-k recommendations using Thompson sampling.
        
        Args:
            k: Number of items to recommend
            exploit: If True, use posterior mean instead of sampling (pure exploitation)
        
        Returns:
            List of (item_id, score) tuples sorted by score descending
        """
        if exploit:
            scores = [p.mean() for p in self.posteriors]
        else:
            scores = [p.sample() for p in self.posteriors]
        
        # Get top k items by score
        items_and_scores = list(enumerate(scores))
        top_k = sorted(items_and_scores, key=lambda x: x[1], reverse=True)[:k]
        return top_k
    
    def update(self, item_id: int, success: bool) -> None:
        """Update an item's posterior with new feedback."""
        self.posteriors[item_id].update(success)

In [ ]:
# Create recommender with 100 items
n_items = 100
rec = ThompsonSamplingRecommender(n_items)

# Simulate some "true" item qualities (probability of success)
rng = np.random.RandomState(42)
true_qualities = rng.beta(2, 5, size=n_items)  # mostly low quality, some gems

# Run for 1000 iterations
n_iter = 1000
cumulative_reward = 0

print("Starting Thompson Sampling simulation...")
print("\nFirst 5 recommendations (cold-start):")
initial_recs = rec.recommend(k=5)
for i, (item_id, score) in enumerate(initial_recs, 1):
    print(f"{i}. Item {item_id}: score={score:.3f} (true quality={true_qualities[item_id]:.3f})")

# Simulation loop
for t in range(n_iter):
    # Get top recommendation
    [(item_id, _)] = rec.recommend(k=1)
    
    # Simulate user feedback
    success = rng.random() < true_qualities[item_id]
    cumulative_reward += success
    
    # Update model
    rec.update(item_id, success)

print(f"\nAfter {n_iter} iterations:")
print(f"Average reward: {cumulative_reward/n_iter:.3f}") # We should see this improve over time, better than uniform random generator with 2/(2+5)=0.285 average reward.

print("\nFinal top 5 recommendations:")
final_recs = rec.recommend(k=5)
for i, (item_id, score) in enumerate(final_recs, 1):
    post = rec.posteriors[item_id]
    print(f"{i}. Item {item_id}: score={score:.3f}, E[quality]={post.mean():.3f} "
            f"(true={true_qualities[item_id]:.3f}), n_obs={post.alpha + post.beta - 2:.0f}")

Starting Thompson Sampling simulation...

First 5 recommendations (cold-start):
1. Item 68: score=0.989 (true quality=0.132)
2. Item 89: score=0.984 (true quality=0.184)
3. Item 64: score=0.957 (true quality=0.114)
4. Item 4: score=0.920 (true quality=0.550)
5. Item 57: score=0.916 (true quality=0.343)

After 1000 iterations:
Average reward: 0.476

Final top 5 recommendations:
1. Item 50: score=0.822, E[quality]=0.286 (true=0.366), n_obs=5
2. Item 67: score=0.780, E[quality]=0.777 (true=0.801), n_obs=334
3. Item 32: score=0.747, E[quality]=0.500 (true=0.310), n_obs=8
4. Item 76: score=0.685, E[quality]=0.565 (true=0.460), n_obs=21
5. Item 70: score=0.683, E[quality]=0.609 (true=0.337), n_obs=21
